In [ ]:
import os
import sys

from langgraph.graph import END, START, MessagesState, StateGraph

sys.path.append(os.path.abspath(os.path.join("..")))
from typing import cast

from IPython.display import Image, display
from langchain_core.messages import HumanMessage, RemoveMessage, SystemMessage
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.memory import MemorySaver

from utils.model import Models


In [ ]:
model = Models()
llm = model.create_groq_model()

In [ ]:
class State(MessagesState, total=False):
    summary: str

In [ ]:
def call_llm(state: State):
    """
    Call the LLM model and pass the summary  and message
    And if no summary pass the messages in the state
    """
    summary = state.get("summary")
    if summary:
        sys_msg = f"Here is the summary of the conversation so far {summary}\n\n"
        messages = [SystemMessage(content=sys_msg)] + state["messages"]
    else:
        messages = state["messages"]
    response = llm.invoke(messages)
    return {"messages": response}

In [ ]:
def summarize_conversation(state: State):
    """Summarize the conversation and store that in summary and trim the messages in the state except the recent two"""
    summary = state.get("summary")
    if summary:
        input_msg = (
            f"Here is the summary of the conversation so far {summary}"
            "Take the new messages into consideration when reading the summary and summarize the conversation"
        )
        messages = [HumanMessage(input_msg)] + state["messages"]
    else:
        input_msg = "Create the summary of the conversation"
        messages = [HumanMessage(input_msg)] + state["messages"]

    response = llm.invoke(messages)
    deleted_messages = [
        RemoveMessage(id=cast(str, m.id)) for m in state["messages"][:-2]
    ]

    return {"summary": response.content, "messages": deleted_messages}

In [ ]:
def should_summarize(state: State):
    messages = state.get("messages")
    if len(messages) > 6:
        return "summarize_conversation"
    else:
        return END

In [ ]:
memory = MemorySaver()
builder = StateGraph(MessagesState)
builder.add_node("call_llm", call_llm)
builder.add_node("summarize_conversation", summarize_conversation)
builder.add_edge(START, "call_llm")
# INFO: Add Conditional edege and map the routing path, that is returned from the  conditional router
builder.add_conditional_edges(
    "call_llm", should_summarize, ["summarize_conversation", END]
)

builder.add_edge("summarize_conversation", END)
workflow = builder.compile(checkpointer=memory)

display(Image(workflow.get_graph().draw_mermaid_png()))

In [ ]:
input_config: RunnableConfig = {"configurable": {"thread_id": "1"}}

In [ ]:
input_msg = HumanMessage(content="Hi")
output = workflow.invoke({"messages": [input_msg]}, config=input_config)
for m in output["messages"]:
    m.pretty_print()


In [ ]:
input_msg = HumanMessage(content="I am Jason")
output = workflow.invoke({"messages": [input_msg]}, config=input_config)
for m in output["messages"]:
    m.pretty_print()

In [ ]:
input_msg = HumanMessage(content="I like football")
output = workflow.invoke({"messages": [input_msg]}, config=input_config)
for m in output["messages"]:
    m.pretty_print()

In [ ]:
input_msg = HumanMessage(content="I like messi and ronaldo")
output = workflow.invoke({"messages": [input_msg]}, config=input_config)
for m in output["messages"]:
    m.pretty_print()
